In [1]:
from transformers import AutoModelForCausalLM
import torch

import os
DEVICE = 2
os.environ["CUDA_VISIBLE_DEVICES"] = str(DEVICE)
torch.set_default_device("cuda:0")

model_id = "nordavind-8bit-5600steps"
model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", load_in_8bit=True
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    padding_side="left",
    add_eos_token=True,
    add_bos_token=True,
)
tokenizer.pad_token = tokenizer.eos_token

In [4]:
prompt = """Gitt artikkelen:
Justisdepartementet bestilte oppgraderinger i kontorlokalene for 299 millioner kroner. Oppgraderingene ble finansiert av daværende eier av bygget, Storebrand, skriver DN.
– Statsbygg inngikk en ordinær leiekontrakt for Gullhaug Torg med et tillegg for investeringer som skulle inngå i husleien, skriver kommunikasjonsdirektør Hege Njaa Aschim i Statsbygg til DN.

DN har stilt spørsmål til Statsbygg og Justisdepartementet om leieavtalen, og fått svar av Statsbygg.

Statsbygg mener Justisdepartmentets leieavtale skiller seg vesentlig fra det NSM har blitt kritisert for:

– Ja, absolutt. Staten betaler en husleie som inkluderer alt, også oppgradering og tilpasninger. Kostnader ved erstatningslokaler til departementene, inkludert Gullhaug Torg, er forankret i Stortinget, med tilhørende bevilgninger. Staten har ikke inngått noen låneavtale knyttet til Gullhaug torg, slik NSM har gjort for sine lokaler. For ordens skyld opplyser vi om at det er Statsbygg som har inngått leieavtalen på vegne av staten, skriver Aschim til DN.
Oppsummer den i én setning:
"""

model_inputs = tokenizer(prompt, return_tensors="pt")
generated_ids = model.generate(**model_inputs, max_new_tokens=1024)
tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


KeyboardInterrupt: 

# pipeline

In [1]:
from transformers import pipeline

#pipe = pipeline("text-generation", model="tollefj/nordavind-8bit-4096", device=1)
pipe = pipeline("text-generation", model="nordavind-8bit-2500steps", device=2)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [2]:
def make_prompt(inst, inp="", sys='Du er "Nordavind", en hjelpsom assistent.'):
    q = f"{inst} {inp} ".strip()
    return f"""<s>{sys} [INST] {q} [/INST] \n"""

In [8]:
q = "Gitt følgende referanser til programmeringsspråk: 1) en type slange, 2) en indonesisk øy, 3) en rød edelsten, og 4) en reaksjon mellom grunnstoffene Fe og O. Hvilke språk er det?"
q = "Løs følgende gåte: 1) en type slange, 2) en indonesisk øy og en type kaffe, 3) en rød edelsten, og 4) oksidasjon av elementet Fe. Lag en liste over tingene jeg tenker på og konkluder med en kategori som passer for alle."
q = "Løs følgende gåte: 1) en type slange, 2) en indonesisk øy og en type kaffe, 3) en rød edelsten, og 4) oksidasjon av elementet Fe. Lag en liste over tingene jeg tenker på. De relaterer til datamaskiner på en eller annen måte."

q = """Gitt følgende artikkel:
– Investeringsleige er vanleg både for offentlege og private leigetakarar. I takt med investeringa i lokala, betalar leigetakar auka leigepris, som er avtalt på førehand.

Det seier Dag-Jørgen Saltnes, som er redaktør i Estate Nyheter, ein nettstad for eigedomsbransjen.

Han meiner at det ikkje er eit klart skilje mellom såkalla investeringsleige og eit lån.

– Har ikkje inngått investeringsleige
NRK er kjent med at det er fleire offentlege verksemder som har avtalar med innebygde finansieringar.

Samstundes seier dei offentlege verksemdene NRK har fått kontakt med at det er vesensforskjellar mellom avtalane dei har og avtalen til NSM.

Nav, som har vore i eit leigeforhold med Norwegian property sidan 2016, seier avtalen er ein «ordinær leigeavtale».

– Leigeforholdet og kontrakten inneheld ingen lånefinansierte forhold. Vi har ikkje inngått såkalla investeringsleige slik det kjem fram i NSM-saka, seier Marianne Fålun som er økonomi- og styringsdirektør i Nav.
""".replace("\n", " ").strip()
q += " Kan du gi en oppsummering?"




q="Hva kan du fortelle meg om Harstad kommune?"

print(q)
# text = pipe(make_prompt(q.strip()), max_length=200, repetition_penalty=1.1, return_text=False)

def gen(q, max_len=400, rep_pen=1.1, temp=0.7, sample=True):
    text = pipe(make_prompt(q.strip()), max_length=max_len, repetition_penalty=rep_pen, temperature=temp, return_text=False, do_sample=sample)
    return text[0]["generated_text"].split("[/INST]")[-1].strip()

gen(q, max_len=1024, temp=0.8, sample=True)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Hva kan du fortelle meg om Harstad kommune?


'Harstad, tatt i bruk som byfjell i midten av 1700-tallet, ble en fullt utviklet by i slutten av 1800-tallet og fikk kunnskap om et navn på samme tidspunkt. Den var særegen for sine karakteristiske lampeposter. I begynnelsen av første verdenskrig ble mange av dem revet ned for å gjenbrukes som ammunisjon, og andre ble fortrykt under bombardementene. Av de 36 økende lampepostene blir det enn i dag og ligger i fjerne enden av byen. Etter at Harstad ble en viktig del av den tyske besatte Norge i 1942 ble flere bygninger totalt ødelagt under de tyske invasjonene sent i 1944 og begynten av 1945. På grunn av sitt beliggenhet over for skjærket kjente som Vardø havn, er Harstad ofte utsatt for alvorlige storme, selv om det også er en veldig stille fylle med sine fina strandene og utsiktssteder. Vanskelig å definere geografisk, deler Harstad kommunen Nord-Troms Fylke i tre: Harstad byfylke på vestbredden, Kvaløyens nordøstre fylke og Magerøya kommunen på øster side. Harstad byfylke har omtrent 